# Analisador de Contratos Imobiliários
**RAG + Saídas Estruturadas com GPT-4o-mini**

## 1. Instalação de dependências

In [ ]:
%pip install --quiet --upgrade \
    langchain langchain-community langchain-openai langchain-core \
    langchain-text-splitters langgraph pypdf python-dotenv

## 2. Configuração da API Key

In [ ]:
import os

try:
    # Colab: chave salva em Secrets
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Chave carregada do Colab Secrets.")
except ImportError:
    # Local: carrega do .env
    from dotenv import load_dotenv
    load_dotenv()
    print("Chave carregada do .env.")

## 3. Imports

In [ ]:
import json
import tempfile
from typing import List

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import TypedDict

## 4. Schemas e Queries

Define a estrutura esperada de saída (JSON) e as queries de busca por seção.

In [ ]:
SCHEMA_CONTRATO = {
    "comprador": {
        "nome_completo": "",
        "cpf": "",
        "rg": "",
        "nacionalidade": "",
        "estado_civil": "",
        "conjuge": {"nome": "", "cpf": "", "rg": ""},
        "profissao": "",
        "endereco": "",
        "email": ""
    },
    "vendedor": {
        "nome_completo": "",
        "cnpj": "",
        "endereco_sede": "",
        "representantes_legais": []
    },
    "imovel": {
        "endereco_completo": "",
        "matricula": "",
        "cartorio_registro": "",
        "inscricao_iptu": "",
        "area_total_m2": "",
        "areas_lazer": "",
        "comodos": "",
        "vagas_garagem": ""
    },
    "condicoes_financeiras": {
        "preco_total": "",
        "indice_reajuste": "",
        "periodicidade_reajuste": "",
        "juros_mora": "",
        "multa_atraso": "",
        "comissao_corretagem": ""
    }
}

QUERIES = {
    "comprador":             "nome completo CPF RG nacionalidade estado civil cônjuge profissão endereço email comprador",
    "vendedor":              "vendedor incorporadora CNPJ endereço sede representantes legais",
    "imovel":                "imóvel matrícula cartório registro IPTU área total lazer cômodos vagas garagem endereço",
    "condicoes_financeiras": "preço total valor venda índice reajuste INCC IGPM IPCA juros multa atraso comissão corretagem"
}

## 5. Prompt

Instrui o LLM a extrair apenas dados do papel solicitado, sem confundir comprador com vendedor.

In [ ]:
PROMPT = ChatPromptTemplate.from_messages([
    ("human", """Você é um especialista em análise de contratos jurídicos brasileiros.

Analise os trechos do contrato abaixo e extraia SOMENTE as informações do papel solicitado na questão.

REGRAS DE VALIDAÇÃO DE PAPEL:
- Se a questão pede dados do COMPRADOR: extraia apenas campos explicitamente atribuídos ao COMPRADOR no texto. Ignore qualquer dado do VENDEDOR/INCORPORADORA.
- Se a questão pede dados do VENDEDOR/INCORPORADORA: extraia apenas campos explicitamente atribuídos ao VENDEDOR ou INCORPORADORA. Ignore dados do COMPRADOR.
- Se a questão pede dados do IMÓVEL: extraia apenas características e identificação do imóvel. Ignore dados das partes.
- Se a questão pede condições financeiras: extraia apenas valores, índices e penalidades contratuais. Ignore dados pessoais.
- Extraia APENAS o que estiver explicitamente no texto. Não infira nem complete com suposições.
- Para campos não encontrados, use string vazia \"\" ou lista vazia [].
- Retorne SOMENTE um objeto JSON válido. Sem explicações fora do JSON.

Schema esperado:
{schema}

Question: {question}
Context: {context}
Answer:""")
])

## 6. Funções do Pipeline RAG

In [ ]:
def build_vector_store(pdf_path: str):
    """
    Carrega o PDF, divide em chunks e indexa no vector store.
    Retorna (vector_store, all_splits).
    """
    embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
    vector_store = InMemoryVectorStore(embeddings)

    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print(f"PDF carregado: {len(docs)} página(s)")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        add_start_index=True,
    )
    all_splits = splitter.split_documents(docs)
    print(f"Chunks gerados: {len(all_splits)}")

    vector_store.add_documents(all_splits)
    print("Vector store indexado.")
    return vector_store, all_splits


def _search_terms(value) -> list:
    """Extrai tokens significativos de qualquer tipo de valor extraído."""
    if isinstance(value, dict):
        texts = [str(v) for v in value.values() if v and str(v).strip()]
    elif isinstance(value, list):
        texts = [str(v) for v in value if v and str(v).strip()]
    else:
        texts = [str(value)] if value else []

    tokens = set()
    for text in texts:
        for token in text.split():
            if len(token) > 3 or any(c.isdigit() for c in token):
                tokens.add(token.lower())
    return list(tokens)


def find_source_chunks(value, all_splits: list, k: int = 2) -> list:
    """
    Localiza os chunks que contêm o valor extraído via contagem de tokens.
    Mais preciso que busca semântica para atribuição de fontes.
    """
    terms = _search_terms(value)
    if not terms:
        return []

    scored = []
    for split in all_splits:
        content_lower = split.page_content.lower()
        hits = sum(1 for t in terms if t in content_lower)
        if hits > 0:
            scored.append((hits, split))

    if not scored:
        return []

    scored.sort(key=lambda x: x[0], reverse=True)
    max_hits = scored[0][0]
    threshold = max(1, max_hits - 1)
    return [s for hits, s in scored[:k] if hits >= threshold]

## 7. Pipeline Principal (LangGraph)

Grafo com dois nós: `retrieve → generate`

In [ ]:
def build_rag_graph(llm, vector_store):
    class State(TypedDict):
        question: str
        schema: str
        context: List[Document]
        answer: str

    def retrieve(state: State):
        return {"context": vector_store.similarity_search(state["question"], k=4)}

    def generate(state: State):
        docs_content = ""
        for i, doc in enumerate(state["context"], 1):
            pos = doc.metadata.get("start_index", "?")
            docs_content += f"[Trecho {i} — posição {pos} no documento]\n{doc.page_content}\n\n"

        messages = PROMPT.invoke({
            "question": state["question"],
            "context":  docs_content,
            "schema":   state["schema"]
        })
        return {"answer": llm.invoke(messages).content}

    graph = StateGraph(State).add_sequence([retrieve, generate])
    graph.add_edge(START, "retrieve")
    return graph.compile()

## 8. Upload e Execução

Faça upload do contrato em PDF e execute a análise.

In [ ]:
# --- Upload do arquivo (Colab) ---
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"Arquivo: {pdf_path}")
except ImportError:
    # Local: informe o caminho do PDF
    pdf_path = "contrato.pdf"
    print(f"Usando arquivo local: {pdf_path}")

In [ ]:
# --- Indexação do PDF ---
vector_store, all_splits = build_vector_store(pdf_path)

In [ ]:
# --- Análise por seção ---
llm = init_chat_model("gpt-4o-mini", model_provider="openai")
graph = build_rag_graph(llm, vector_store)

resultado_final = {}

for secao, query in QUERIES.items():
    print(f"\nAnalisando: {secao}...")

    result = graph.invoke({
        "question": f"Extraia as informações de '{secao}' do contrato: {query}",
        "schema":   json.dumps(SCHEMA_CONTRATO[secao], ensure_ascii=False, indent=2)
    })

    raw = result["answer"].replace("```json", "").replace("```", "").strip()
    try:
        dados = json.loads(raw)
    except json.JSONDecodeError:
        dados = raw

    trechos_por_campo = {}
    if isinstance(dados, dict):
        for field, field_value in dados.items():
            source_docs = find_source_chunks(field_value, all_splits, k=2)
            trechos_por_campo[field] = [
                {
                    "trecho":  doc.page_content,
                    "posicao": doc.metadata.get("start_index", "?"),
                    "pagina":  doc.metadata.get("page", "?")
                }
                for doc in source_docs
            ]

    resultado_final[secao] = {
        "dados":            dados,
        "trechos_por_campo": trechos_por_campo
    }

print("\nAnálise concluída!")

## 9. Exibição dos Resultados

In [ ]:
from IPython.display import display, HTML
import html as html_lib

FIELD_LABELS = {
    "nome_completo":         "Nome Completo",
    "cpf":                   "CPF",
    "rg":                    "RG",
    "nacionalidade":         "Nacionalidade",
    "estado_civil":          "Estado Civil",
    "conjuge":               "Cônjuge",
    "profissao":             "Profissão",
    "endereco":              "Endereço",
    "email":                 "E-mail",
    "cnpj":                  "CNPJ",
    "endereco_sede":         "Endereço da Sede",
    "representantes_legais": "Representantes Legais",
    "endereco_completo":     "Endereço Completo",
    "matricula":             "Matrícula",
    "cartorio_registro":     "Cartório de Registro",
    "inscricao_iptu":        "Inscrição IPTU",
    "area_total_m2":         "Área Total (m²)",
    "areas_lazer":           "Áreas de Lazer",
    "comodos":               "Cômodos",
    "vagas_garagem":         "Vagas de Garagem",
    "preco_total":           "Preço Total",
    "indice_reajuste":       "Índice de Reajuste",
    "periodicidade_reajuste":"Periodicidade do Reajuste",
    "juros_mora":            "Juros de Mora",
    "multa_atraso":          "Multa por Atraso",
    "comissao_corretagem":   "Comissão de Corretagem"
}

SECTION_LABELS = {
    "comprador":             "Comprador",
    "vendedor":              "Vendedor / Incorporadora",
    "imovel":                "Imóvel",
    "condicoes_financeiras": "Condições Financeiras"
}

def render_value(v) -> str:
    if isinstance(v, dict):
        parts = [f"{k}: {val}" for k, val in v.items() if val]
        return ", ".join(parts) if parts else ""
    if isinstance(v, list):
        return "; ".join(str(x) for x in v) if v else ""
    return str(v) if v else ""

def highlight_value(value: str, text: str, context: int = 200) -> str:
    if not value or not text:
        return html_lib.escape(text[:400] if text else "")
    text_lower = text.lower()
    term = value.strip()
    pos = text_lower.find(term.lower())
    if pos == -1:
        tokens = sorted(
            [t for t in term.split() if len(t) > 4 or any(c.isdigit() for c in t)],
            key=len, reverse=True
        )
        for token in tokens:
            p = text_lower.find(token.lower())
            if p != -1:
                pos, term = p, token
                break
    if pos == -1:
        return html_lib.escape(text[:400])
    ws = max(0, pos - context)
    we = min(len(text), pos + len(term) + context)
    window = text[ws:we]
    mp = window.lower().find(term.lower())
    if mp != -1:
        return (
            ("..." if ws > 0 else "") +
            html_lib.escape(window[:mp]) +
            '<mark style="background:#fef08a;color:#1e293b;padding:1px 4px;border-radius:3px;font-weight:700">' +
            html_lib.escape(window[mp:mp+len(term)]) +
            "</mark>" +
            html_lib.escape(window[mp+len(term):]) +
            ("..." if we < len(text) else "")
        )
    return html_lib.escape(window)

def render_resultado(resultado: dict):
    for secao, conteudo in resultado.items():
        dados = conteudo["dados"]
        trechos_por_campo = conteudo["trechos_por_campo"]
        titulo = SECTION_LABELS.get(secao, secao)

        html = f"""
        <h2 style="border-bottom:2px solid #3b82f6;padding-bottom:6px;color:#93c5fd">{titulo}</h2>
        <table style="width:100%;border-collapse:collapse;font-family:sans-serif;font-size:14px;color:#e2e8f0">
          <thead>
            <tr style="background:#1e293b">
              <th style="text-align:left;padding:8px 12px;width:30%;color:#e2e8f0">Campo</th>
              <th style="text-align:left;padding:8px 12px;color:#e2e8f0">Valor</th>
            </tr>
          </thead>
          <tbody>
        """

        if isinstance(dados, dict):
            for field, value in dados.items():
                label = FIELD_LABELS.get(field, field)
                rendered = render_value(value)
                valor_html = html_lib.escape(rendered) if rendered else "<em style='color:#64748b'>Não encontrado</em>"

                trechos = trechos_por_campo.get(field, [])
                trecho_html = ""
                if rendered and trechos:
                    trecho_html = "<br>"
                    for i, t in enumerate(trechos, 1):
                        hl = highlight_value(rendered, t["trecho"])
                        trecho_html += f"""
                        <details style="margin-top:6px">
                          <summary style="cursor:pointer;color:#60a5fa;font-size:12px">
                            Trecho {i} — Página {t['pagina']} · posição {t['posicao']}
                          </summary>
                          <pre style="background:#0f172a;border:1px solid #334155;border-radius:4px;
                                      padding:10px;font-size:12px;white-space:pre-wrap;
                                      line-height:1.6;margin-top:4px;color:#e2e8f0">{hl}</pre>
                        </details>
                        """

                html += f"""
                <tr style="border-bottom:1px solid #334155">
                  <td style="padding:10px 12px;font-weight:500;color:#94a3b8;vertical-align:top">{label}</td>
                  <td style="padding:10px 12px;vertical-align:top;color:#e2e8f0">{valor_html}{trecho_html}</td>
                </tr>
                """

        html += "</tbody></table><br>"
        display(HTML(html))

render_resultado(resultado_final)

## 10. Resultado bruto (JSON)

In [ ]:
for secao, conteudo in resultado_final.items():
    print(f"\n{'='*60}")
    print(f"  {SECTION_LABELS.get(secao, secao).upper()}")
    print(f"{'='*60}")
    print(json.dumps(conteudo["dados"], ensure_ascii=False, indent=2))